### Data Source
**Dataset:** Online Retail II \
**Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

### Dataset Description
This dataset contains all the transactions occurring for a UK-based and registered, non-store online retailer between 01/12/2009 and 09/12/2011. The company mainly sells unique all-occasion gift-ware, and many of its customers are wholesalers. The data spans two full years and is provided as two Excel sheets, one per year.

The dataset contains 1,067,371 instances across 8 features, characterized as multivariate, sequential, time-series data with both integer and real-valued fields. It is associated with three types of machine learning tasks: Classification, Regression, and Clustering, which is directly relevant to this project's planned modeling approach (CLV regression, churn classification, customer segmentation via clustering).

**Columns:**
- `Invoice` — 6-digit invoice number; a prefix of 'C' indicates a cancelled transaction
- `StockCode` — 5-digit product code
- `Description` — product name
- `Quantity` — units purchased per transaction
- `InvoiceDate` — date and time of transaction
- `Price` — unit price in GBP (£)
- `Customer ID` — 5-digit customer identifier
- `Country` — customer's country of residence


In [1]:
import pandas as pd
import sys
sys.path.append("../src")
from utils import report_reduction

# List sheet names in the source Excel file to confirm its structure before loading
pd.ExcelFile("../data/online_retail_II.xlsx").sheet_names

['Year 2009-2010', 'Year 2010-2011']

In [2]:
# Load each yearly sheet separately
df_2009_2010 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2010-2011")

# Combine both years into a single dataframe covering the full date range
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

# Confirm the combined row count matches UCI's stated total (1,067,371 instances)
df.shape

(1067371, 8)

### Check Data Types and Missing Values

With the full dataset loaded, we inspect column data types and non-null counts. This tells us which columns have missing values (and how many) before deciding how to handle them, and confirms whether any columns need dtype conversion.

In [3]:
# Check dtypes and non-null counts per column to identify missing values and confirm data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


### Quantify Missing Values

`df.info()` shows non-null counts, but it's clearer to explicitly compute the count and percentage of missing values per column. This confirms `Description` and `Customer ID` are the only columns with missing data, and shows the scale of each — which will inform how we handle them.

In [4]:
# Compute missing value count and percentage per column
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})

# Only show columns that actually have missing values
missing_summary[missing_summary["Missing Count"] > 0]

,Missing Count,Missing %
Description,4382,0.41
Customer ID,243007,22.77


### Check for Duplicate Rows

Before investigating specific data quality issues like cancellations or invalid values, we check for fully duplicated rows. Duplicate transaction records — whether from data entry errors or export artifacts — could distort Frequency and Monetary calculations later if left unaddressed.

In [5]:
# Count fully duplicated rows across all columns
df.duplicated().sum()

np.int64(34335)

Duplicate rows account for a meaningful portion of the dataset (~3.2%). Before deciding whether to remove them, we inspect a sample to understand whether these represent genuine repeated line items (e.g., the same product added twice to an order) or artifacts from data export/loading.

In [6]:
# Inspect a sample of duplicated rows to understand their nature before deciding how to handle them
df[df.duplicated(keep=False)].sort_values(by=["Invoice", "StockCode"]).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [7]:
# Remove exact duplicate rows, keeping the first occurrence
df = df.drop_duplicates().reset_index(drop=True)

In [8]:
# Confirm the new shape after removing duplicates
df.shape

(1033036, 8)

### Investigate Cancellations

Per the dataset documentation, an `Invoice` value prefixed with 'C' indicates a cancelled transaction. We quantify how many such rows exist before deciding how to handle them.

In [9]:
# Count rows flagged as cancellations (Invoice starts with 'C')
df['Invoice'].astype(str).str.startswith('C').sum()

np.int64(19104)

### Compare Cancellations to Negative Quantity Rows

Cancellations are identified by the 'C' prefix in `Invoice`, but negative `Quantity` values could also indicate returns, adjustments, or other non-purchase entries that aren't captured by this prefix alone. We compare the two to check for overlap and identify any additional rows that need investigation.

In [10]:
# Count rows with negative Quantity
negative_qty = (df['Quantity'] < 0).sum()

# Count rows flagged as cancellations
cancellations = df['Invoice'].astype(str).str.startswith('C').sum()

# Count rows with negative Quantity that are NOT flagged as cancellations
negative_qty_not_cancelled = ((df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))).sum()

print("Negative Quantity rows:", negative_qty)
print("Cancellation rows (Invoice starts with C):", cancellations)
print("Negative Quantity but NOT flagged as cancellation:", negative_qty_not_cancelled)

Negative Quantity rows: 22496
Cancellation rows (Invoice starts with C): 19104
Negative Quantity but NOT flagged as cancellation: 3393


As before, we inspect the `StockCode` values for rows with negative `Quantity` that are not flagged as cancellations, to understand whether these represent a distinct pattern (e.g., manual adjustment codes) or scattered data inconsistencies.

In [11]:
# Inspect StockCode values for negative-Quantity rows not flagged as cancellations
df[(df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))]['StockCode'].value_counts().head(15)

StockCode
22423     11
82494L     6
22719      6
46000M     5
84016      5
85017A     5
47566B     5
20852      5
85175      5
21830      5
71477      4
46000S     4
21768      4
84559D     4
84990      4
Name: count, dtype: int64

These rows show scattered, real-looking product stock codes with no dominant adjustment-code pattern (e.g., no concentration in codes like "D", "M", "POST", or "BANK CHARGES"). This confirms these are likely unflagged returns or data entry inconsistencies rather than a distinct category requiring separate handling. We proceed with the same cleaning rule as cancellations: filtering on `Quantity > 0` captures both cases in one consistent step.

### Check for Invalid Price Values

Beyond Quantity issues, we check for rows with zero or negative `Price`, which would not represent genuine revenue-generating transactions.

In [12]:
# Count rows with zero or negative Price
(df['Price'] <= 0).sum()

np.int64(6019)

### Check for Non-Product StockCode Entries

Beyond missing values, duplicates, and invalid Quantity/Price, some rows have a `StockCode` that doesn't represent a real product — administrative or adjustment entries rather than merchandise. We check for known non-product codes: `POST` (postage), `M` (manual entries), `C2` (secondary cancellations), `BANK CHARGES`, `DOT`, and `D` (discounts).

In [13]:
non_product_codes = ['POST', 'M', 'C2', 'BANK CHARGES', 'DOT', 'D']

df['StockCode'].value_counts().loc[lambda x: x.index.isin(non_product_codes)]

StockCode
POST            2086
DOT             1425
M               1387
C2               277
D                173
BANK CHARGES     100
Name: count, dtype: int64

These non-product codes account for a small but meaningful number of rows. Since they represent administrative entries (postage, manual adjustments, fees) rather than genuine merchandise sales, they should be excluded from the final cleaned dataset alongside the other filters already identified.

### Apply Final Cleaning Filters

Based on the checks above, we apply three filtering rules to produce a clean dataset suitable for customer-level analysis:

- **Drop rows with missing `Customer ID`** : these transactions cannot be attributed to a specific customer and are unusable for CLV calculation.
- **Drop rows with `Quantity <= 0`** : this captures both formal cancellations and unflagged negative-quantity entries in a single consistent rule.
- **Drop rows with `Price <= 0`** : these do not represent genuine revenue-generating transactions.

We compare the shape before and after filtering to quantify the impact of cleaning.

In [14]:
before = df.shape

df_clean = df[
    (df['Customer ID'].notna()) &
    (df['Quantity'] > 0) &
    (df['Price'] > 0) &
    (~df['StockCode'].isin(non_product_codes))
].copy()

report_reduction(before, df_clean.shape, step_name="Applying final cleaning filters")

Applying final cleaning filters:
  Before: 1,033,036 rows
  After:  776,641 rows
  Removed: 256,395 rows (24.82%)


In [15]:
# Confirm no missing values remain and check dtypes
df_clean.info()

<class 'pandas.DataFrame'>
Index: 776641 entries, 0 to 1033034
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      776641 non-null  object        
 1   StockCode    776641 non-null  object        
 2   Description  776641 non-null  object        
 3   Quantity     776641 non-null  int64         
 4   InvoiceDate  776641 non-null  datetime64[us]
 5   Price        776641 non-null  float64       
 6   Customer ID  776641 non-null  float64       
 7   Country      776641 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 53.3+ MB


In [16]:
# Confirm Quantity and Price are both strictly positive
df_clean['Quantity'].min(), df_clean['Price'].min()

(np.int64(1), np.float64(0.001))

### Summary

Starting from 1,067,371 raw transaction records, the following cleaning steps were applied:

| Step | Rows Removed | Resulting Shape |
|---|---|---|
| Remove exact duplicate rows | 34,335 | (1,033,036, 8) |
| Drop missing `Customer ID`, `Quantity <= 0`, `Price <= 0`, non-product `StockCode` | 256,395 | (776,641, 8) |

**Total rows removed:** 290,730 (~27.2% of the original dataset)

The cleaned dataset (`df_clean`) contains 776,641 transaction records across 8 columns, with no missing values, strictly positive `Quantity` and `Price` fields, and no non-product administrative entries. This dataset is saved to disk as a checkpoint for downstream feature engineering and modeling.

In [17]:
# Save the cleaned dataset as a checkpoint for downstream notebooks
df_clean.to_csv("../data/online_retail_clean.csv", index=False)